# The GAE-$\lambda$ Bias/Variance Dial

*A sibling notebook to the simple-RL companion series. CPU-only, runs in a few minutes on free Colab.*

Every advantage estimate in policy-gradient RL faces the same tension: do you trust your **value
function** (low variance, but biased if the critic is wrong) or do you trust the **actual returns
you sampled** (unbiased, but high variance)? **Generalized Advantage Estimation (GAE)** gives you
a single dial, $\lambda \in [0,1]$, that smoothly interpolates between those two extremes.

This notebook takes **one clean PPO** on **CartPole-v1** and changes **only** $\lambda$, sweeping
$\lambda \in \{0.0,\ 0.5,\ 0.9,\ 0.95,\ 1.0\}$ with $\gamma=0.99$ fixed and *everything else
identical*. We then watch the bias/variance trade-off two ways: in the **learning curves** (how
fast and how reliably each $\lambda$ learns) and in a **direct variance measurement** (freeze one
policy, draw many independent rollouts, and measure how the advantage estimates spread out).

## The dial

Let $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ be the **one-step TD residual**. GAE defines

$$\hat A_t^{\,\text{GAE}(\gamma,\lambda)} \;=\; \sum_{l=0}^{\infty} (\gamma\lambda)^{l}\,\delta_{t+l}.$$

It is an **exponentially-weighted average of all the $n$-step advantage estimators**. Reading off
the two endpoints makes the trade-off concrete:

| $\lambda$ | what $\hat A_t$ becomes | bias | variance |
|---|---|---|---|
| $\lambda=0$ | $\hat A_t = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ &nbsp; (**one-step TD**) | **high** (leans entirely on the critic $V$) | **low** (one reward + a bootstrap) |
| $0 < \lambda < 1$ | a geometric blend of all $n$-step estimators | tunable | tunable |
| $\lambda=1$ | $\hat A_t = \sum_{l\ge0}\gamma^{l} r_{t+l} - V(s_t)$ &nbsp; (**Monte-Carlo**) | **low** (uses real returns; bias only from $\gamma$) | **high** (sums many noisy rewards) |

So $\lambda$ is literally a **bias$\leftrightarrow$variance knob**. The folklore sweet spot is
$\lambda \approx 0.9$–$0.97$ — enough bootstrapping to tame variance, not so much that the
critic's bias dominates. By the end you will have *measured* why.

> Note: $\gamma$ also controls a bias/variance trade-off (effective horizon), but here we **fix
> $\gamma=0.99$** and isolate $\lambda$. GAE deliberately decouples the two.

**Theory companions (Notion):**
[PPO](https://app.notion.com/p/37f95008d76681ae994febc1f7e89aa5) &nbsp;·&nbsp;
[Part 3: Intro to Policy Optimization](https://app.notion.com/p/37895008d7668195b974f3c59beb74f6)

## 0. Setup

Install the (tiny) dependencies. On Colab this takes a few seconds.

In [ ]:
%pip install -q "gymnasium>=0.29" "pygame>=2.5"


### Imports and **global config**

Everything you might want to tweak lives in one place. The whole experiment changes **only**
`LAMBDAS`; `GAMMA`, the networks, the PPO knobs, the seeds and the step budget are shared across
every run, so the sweep is a clean controlled experiment. Small nets + CPU on purpose.

In [ ]:
import time, math, random
from typing import List

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym

# ----- reproducibility helper ------------------------------------------------------------
def set_seed(seed: int):
    """Seed Python, NumPy and Torch so a run is repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

DEVICE = torch.device("cpu")          # free Colab: CPU is plenty for CartPole + tiny nets

# =====================================================================================
#                                 GLOBAL CONFIG
# =====================================================================================
ENV_ID        = "CartPole-v1"         # the shared task
GAMMA         = 0.99                   # discount factor -- FIXED for the whole notebook
LAMBDAS       = [0.0, 0.5, 0.9, 0.95, 1.0]   # the ONLY thing we sweep (the bias/variance dial)
SEEDS         = [0, 1, 2]             # average over a few seeds -> mean +/- std bands
SOLVED_RETURN = 475.0                # CartPole-v1 is "solved" at avg return >= 475 (max 500)

# --- training budget (per lambda, per seed) ---------------------------------------------
ITERS         = 25                    # number of PPO iterations (collect-batch -> optimize)
BATCH         = 2_000                 # env steps collected per on-policy batch (the rollout)

# --- PPO knobs (shared across ALL lambda runs -- only lambda differs) --------------------
PPO_EPOCHS    = 6                     # gradient passes over each fresh batch
PPO_MINIBATCH = 256
PPO_CLIP      = 0.2                   # clipped-surrogate epsilon (trust region)
PPO_LR        = 3e-3
PPO_ENT_COEF  = 0.0                   # entropy bonus (0 is fine for CartPole)
PPO_VF_COEF   = 0.5                   # value-loss weight
PPO_MAX_GRAD  = 0.5                   # global grad-norm clip
NORM_ADV      = True                  # normalize advantages per batch (standard PPO practice)

HIDDEN        = 64                    # hidden width for ALL nets (kept tiny on purpose)

print("torch     :", torch.__version__)
print("gymnasium :", gym.__version__)
print("numpy     :", np.__version__)
print("device    :", DEVICE)
print("sweep     : lambda in", LAMBDAS, "  (gamma fixed =", GAMMA, ")")
print("budget    :", ITERS, "iters x", BATCH, "steps =", ITERS * BATCH,
      "env steps / lambda / seed   seeds =", SEEDS)


## 1. The task: CartPole-v1

A pole is hinged to a cart on a track. The agent pushes the cart **left (0)** or **right (1)**;
the goal is to keep the pole upright. State is 4-D (cart position, cart velocity, pole angle, pole
angular velocity). Reward is **+1 per timestep**; the episode ends when the pole falls or the cart
runs off, capped at **500** steps. "Solved" = average return $\ge 475$.

CartPole is the perfect stage for this demo: returns are bounded and the dynamics are smooth, so
differences between $\lambda$ settings are about *estimator quality*, not task pathology.


## 2. Networks

A small 2-layer MLP **policy** (actor) and a separate 2-layer MLP **value function** (critic),
each of width `HIDDEN`. The critic $V(s)$ is the baseline that GAE leans on — so how much we
trust it is exactly what $\lambda$ controls.

In [ ]:
class PolicyNet(nn.Module):
    """Actor: state -> logits over actions -> Categorical distribution."""
    def __init__(self, obs_dim, n_actions, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return Categorical(logits=self.net(x))    # a torch distribution we can sample/score


class ValueNet(nn.Module):
    """Critic: state -> scalar state-value V(s) (the baseline GAE bootstraps from)."""
    def __init__(self, obs_dim, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)            # shape: (batch,)


## 3. On-policy rollout

Collect a fresh batch of `n_steps` environment steps with the **current** policy, recording
`obs, action, log-prob, reward, value, done` at each step (plus the bootstrap value of the state
we stopped at). This is the raw material GAE turns into advantages. On-policy = we use this batch
once and throw it away.

In [ ]:
def collect_rollout(env, policy, value, obs, n_steps):
    """Run the current policy for `n_steps` env steps starting from `obs`.

    Returns a dict of numpy arrays plus the bootstrap value `last_v` (V of the final state) and
    the observation to resume from. Also returns the list of completed episodic returns so we can
    plot a learning curve. `done` stores the TRUE terminal flag (term, not trunc) so we only stop
    bootstrapping at real episode ends, not at time-limit truncations."""
    b_obs, b_act, b_logp, b_rew, b_val, b_done = [], [], [], [], [], []
    ep_returns, ep_return = [], 0.0

    for _ in range(n_steps):
        ot = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            dist = policy(ot)
            a = dist.sample()
            logp = dist.log_prob(a)
            v = value(ot)
        a_i = int(a.item())
        obs2, r, term, trunc, _ = env.step(a_i)
        done = term or trunc

        b_obs.append(obs); b_act.append(a_i); b_logp.append(float(logp.item()))
        b_rew.append(float(r)); b_val.append(float(v.item())); b_done.append(float(term))

        obs = obs2
        ep_return += r
        if done:
            ep_returns.append(ep_return)
            ep_return = 0.0
            obs, _ = env.reset()

    # bootstrap value for the state we stopped at (used by GAE for the last step)
    with torch.no_grad():
        last_v = float(value(torch.as_tensor(obs, dtype=torch.float32,
                                             device=DEVICE).unsqueeze(0)).item())

    batch = dict(
        obs=np.array(b_obs, dtype=np.float32),
        act=np.array(b_act, dtype=np.int64),
        logp=np.array(b_logp, dtype=np.float32),
        rew=np.array(b_rew, dtype=np.float32),
        val=np.array(b_val, dtype=np.float32),
        done=np.array(b_done, dtype=np.float32),
    )
    return batch, last_v, obs, ep_returns


## 4. The GAE helper, parametrized by $\lambda$

This is the heart of the notebook. The recursion below computes
$\hat A_t = \sum_{l\ge0}(\gamma\lambda)^l \delta_{t+l}$ in one backward pass:

$$\hat A_t = \delta_t + (\gamma\lambda)\,(1-\text{done}_t)\,\hat A_{t+1},\qquad
  \delta_t = r_t + \gamma\,(1-\text{done}_t)\,V(s_{t+1}) - V(s_t).$$

The **only** quantity that changes across our entire sweep is the scalar `lam` fed here. Setting
`lam=0` collapses $\hat A_t \to \delta_t$ (one-step TD); `lam=1` makes it the discounted
Monte-Carlo advantage $\sum_l \gamma^l r_{t+l} - V(s_t)$. The value target is
$\hat R_t = \hat A_t + V(s_t)$ regardless of $\lambda$.

In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma, lam):
    """Generalized Advantage Estimation over one flat rollout (episode boundaries via `dones`).

    rewards, values, dones : 1-D arrays of length T (values are V(s_t)).
    last_value             : V(s_T), the bootstrap for the final step.
    gamma, lam             : the discount and the GAE trace-decay (the bias/variance dial).
    Returns (advantages, returns) where returns = advantages + values (the critic's target)."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(T)):
        next_v = last_value if t == T - 1 else values[t + 1]
        nonterminal = 1.0 - dones[t]            # if episode ended at t, don't bootstrap past it
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]   # one-step TD residual
        last_gae = delta + gamma * lam * nonterminal * last_gae         # the (gamma*lam) trace
        adv[t] = last_gae
    returns = adv + np.asarray(values, dtype=np.float32)
    return adv, returns


## 5. `train(lmbda, seed)` — one clean PPO, lambda as the only argument

Standard PPO-clip with a value baseline and multi-epoch minibatched updates. For `ITERS`
iterations we (1) collect a fresh `BATCH`-step rollout, (2) compute advantages with **this**
`lmbda` via `compute_gae`, (3) optimize the clipped surrogate
$\min(\rho_t\hat A_t,\ \text{clip}(\rho_t,1\pm\epsilon)\hat A_t)$ plus a value loss, then discard
the batch. We log the mean episodic return per iteration so we can draw learning curves.

Every line is shared across the sweep except the `lmbda` passed to `compute_gae` — that is what
makes this a clean isolation of the dial.

In [ ]:
def train(lmbda: float, seed: int, iters: int = ITERS, batch: int = BATCH, verbose: bool = False):
    """Train PPO with GAE trace-decay = `lmbda` (gamma fixed = GAMMA). Returns a curve of mean
    episodic return per iteration and the trained (policy, value) for optional later use."""
    set_seed(seed)
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=seed)
    obs_dim   = env.observation_space.shape[0]
    n_actions = env.action_space.n

    policy = PolicyNet(obs_dim, n_actions).to(DEVICE)
    value  = ValueNet(obs_dim).to(DEVICE)
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=PPO_LR)

    curve = []   # mean episodic return per iteration (the learning curve for this lambda/seed)

    for it in range(iters):
        # ----------------------------------------------------- 1) collect a fresh rollout
        b, last_v, obs, ep_returns = collect_rollout(env, policy, value, obs, batch)

        # ----------------------------------------------------- 2) advantages via GAE(lambda)
        adv, ret = compute_gae(b["rew"], b["val"], b["done"], last_v, GAMMA, lmbda)
        if NORM_ADV:
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)   # standard per-batch normalization

        obs_t  = torch.as_tensor(b["obs"],  dtype=torch.float32, device=DEVICE)
        act_t  = torch.as_tensor(b["act"],  dtype=torch.int64,   device=DEVICE)
        logp_t = torch.as_tensor(b["logp"], dtype=torch.float32, device=DEVICE)
        adv_t  = torch.as_tensor(adv, dtype=torch.float32, device=DEVICE)
        ret_t  = torch.as_tensor(ret, dtype=torch.float32, device=DEVICE)

        # ----------------------------------------------------- 3) PPO multi-epoch update
        N = obs_t.shape[0]
        idx = np.arange(N)
        for _ in range(PPO_EPOCHS):
            np.random.shuffle(idx)
            for start in range(0, N, PPO_MINIBATCH):
                mb = idx[start:start + PPO_MINIBATCH]
                dist = policy(obs_t[mb])
                new_logp = dist.log_prob(act_t[mb])
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - logp_t[mb])             # importance ratio rho_t
                unclipped = ratio * adv_t[mb]
                clipped = torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * adv_t[mb]
                pi_loss = -torch.min(unclipped, clipped).mean()       # clipped surrogate

                v_pred = value(obs_t[mb])
                v_loss = F.mse_loss(v_pred, ret_t[mb])                # value-baseline regression

                loss = pi_loss + PPO_VF_COEF * v_loss - PPO_ENT_COEF * entropy
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(
                    list(policy.parameters()) + list(value.parameters()), PPO_MAX_GRAD)
                opt.step()

        # ----------------------------------------------------- log
        mean_ret = float(np.mean(ep_returns)) if ep_returns else float("nan")
        curve.append(mean_ret)
        if verbose:
            print(f"  iter {it:2d}: mean_ret={mean_ret:6.1f}  (episodes={len(ep_returns)})")

    env.close()
    return np.array(curve, dtype=np.float32), policy, value


## 6. Run the sweep

For each $\lambda$ and each seed we run one full PPO training. The only thing that differs between
the $5\times 3 = 15$ runs is the scalar $\lambda$. On free Colab CPU this is a few minutes.

In [ ]:
def run_sweep(lambdas=LAMBDAS, seeds=SEEDS):
    """Returns curves[lambda] = array of shape (n_seeds, ITERS) of mean episodic return."""
    curves = {lam: [] for lam in lambdas}
    t0 = time.time()
    for lam in lambdas:
        for seed in seeds:
            ts = time.time()
            curve, _, _ = train(lam, seed)
            curves[lam].append(curve)
            final = np.nanmean(curve[-3:]) if len(curve) >= 3 else np.nanmean(curve)
            print(f"lambda={lam:<4} seed={seed}: final~{final:6.1f}  ({time.time()-ts:4.1f}s)")
    for lam in lambdas:
        curves[lam] = np.vstack(curves[lam])
    print(f"\nTOTAL elapsed: {time.time()-t0:.1f}s")
    return curves

curves = run_sweep()


## 7. Plot 1 — learning curves per $\lambda$

Mean $\pm$ std over seeds, with the solved threshold (475) dashed. Reading guide:

- **$\lambda=0$** (pure one-step TD) is the **most biased**: it leans entirely on a critic that is
  poor early on, so it often learns slowly or plateaus low.
- **$\lambda=1$** (Monte-Carlo) is **unbiased** but **high-variance**: curves tend to be jumpier
  and noisier across seeds (wider bands).
- **$\lambda \approx 0.9$–$0.95$** usually learns fastest and most reliably — the sweet spot.

In [ ]:
def smooth(y, k=3):
    if len(y) < k: return y
    return np.convolve(y, np.ones(k) / k, mode="same")

plt.figure(figsize=(9.5, 5.8))
cmap = plt.cm.viridis
its = np.arange(1, ITERS + 1)
finals = {}
for i, lam in enumerate(LAMBDAS):
    M = curves[lam]                                   # (n_seeds, ITERS)
    mean = np.nanmean(M, axis=0); std = np.nanstd(M, axis=0)
    color = cmap(i / max(1, len(LAMBDAS) - 1))
    plt.plot(its, smooth(mean), color=color, lw=2, label=fr"$\lambda$={lam}")
    plt.fill_between(its, smooth(mean) - std, smooth(mean) + std, color=color, alpha=0.15)
    finals[lam] = float(np.nanmean(mean[-3:]))

plt.axhline(SOLVED_RETURN, ls="--", color="gray", lw=1, label=f"solved = {SOLVED_RETURN:.0f}")
plt.xlabel("PPO iteration  (each = one fresh BATCH-step rollout)")
plt.ylabel("mean episodic return")
plt.title(r"Learning curves vs the GAE dial $\lambda$  (CartPole-v1, $\gamma$=0.99, mean$\pm$std over seeds)")
plt.legend(loc="lower right", ncol=2); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("final mean return (last 3 iters) by lambda:")
for lam in LAMBDAS:
    print(f"  lambda={lam:<5}: {finals[lam]:6.1f}")
best_lam = max(finals, key=finals.get)
print(f"\nbest final return at lambda={best_lam} (often in the 0.9-0.95 sweet spot)")


## 8. Plot 2 — the variance, measured directly

The learning curves *hint* at the trade-off; this plot **measures** it. Recipe:

1. Train **one** policy/critic *lightly* (a fixed, shared, partially-trained snapshot — so the
   critic is imperfect, which is exactly when $\lambda$ matters).
2. **Freeze** it. Then for each $\lambda$, draw **many independent small rollouts** from that
   *same* frozen policy and, for each rollout, compute the GAE advantages.
3. Measure the **variance of the advantage estimates across rollouts**, and also the variance of
   the resulting **policy-gradient vector** $\nabla_\theta \log\pi(a|s)\,\hat A$ across rollouts.

Because the policy is frozen and shared, the *only* thing that changes the spread is $\lambda$.
We expect the variance to **rise as $\lambda \to 1$** (more reliance on noisy sampled returns),
confirming the dial.

In [ ]:
def freeze_lightly_trained(seed=0, light_iters=4):
    """Train a single PPO snapshot for a few iters (lambda value here is irrelevant -- we only
    want a shared, imperfect critic to expose the estimator variance), then return it frozen."""
    curve, policy, value = train(lmbda=0.95, seed=seed, iters=light_iters)
    for p in policy.parameters(): p.requires_grad_(True)   # keep grads on for the PG-vector test
    return policy, value


def advantage_variance_vs_lambda(policy, value, lambdas=LAMBDAS, n_rollouts=40,
                                  rollout_len=200, seed=0):
    """For each lambda, draw `n_rollouts` independent short rollouts from the FROZEN policy and
    measure how much the advantage estimates -- and the policy-gradient vector -- vary across
    those rollouts. Returns (adv_var[lam], grad_var[lam])."""
    set_seed(seed)
    env = gym.make(ENV_ID)

    # Collect a bank of fresh rollouts ONCE (same raw experience for every lambda), so the only
    # difference across lambdas is how compute_gae weights it. Each rollout stores per-step
    # rewards/values/dones plus obs/acts (for the gradient test).
    bank = []
    obs, _ = env.reset(seed=seed)
    for _ in range(n_rollouts):
        b, last_v, obs, _ = collect_rollout(env, policy, value, obs, rollout_len)
        bank.append((b, last_v))
    env.close()

    adv_var, grad_var = {}, {}
    for lam in lambdas:
        # ---- (a) variance of the advantage estimates across rollouts -----------------------
        per_rollout_mean_adv = []     # summarize each rollout by its mean |advantage| magnitude
        per_rollout_gradvec  = []     # and by its (flattened) policy-gradient vector
        for (b, last_v) in bank:
            adv, _ = compute_gae(b["rew"], b["val"], b["done"], last_v, GAMMA, lam)
            # use raw (UN-normalized) advantages here -- normalization would hide the variance
            per_rollout_mean_adv.append(float(np.mean(adv)))

            # ---- (b) the policy-gradient vector this rollout would produce ------------------
            obs_t = torch.as_tensor(b["obs"], dtype=torch.float32, device=DEVICE)
            act_t = torch.as_tensor(b["act"], dtype=torch.int64,   device=DEVICE)
            adv_t = torch.as_tensor(adv, dtype=torch.float32, device=DEVICE)
            dist = policy(obs_t)
            logp = dist.log_prob(act_t)
            pg_loss = -(logp * adv_t).mean()          # vanilla PG surrogate with GAE advantages
            policy.zero_grad()
            pg_loss.backward()
            g = torch.cat([p.grad.reshape(-1) for p in policy.parameters() if p.grad is not None])
            per_rollout_gradvec.append(g.detach().cpu().numpy().copy())

        per_rollout_mean_adv = np.array(per_rollout_mean_adv)
        G = np.vstack(per_rollout_gradvec)            # (n_rollouts, n_params)

        adv_var[lam]  = float(np.var(per_rollout_mean_adv))
        # variance of the gradient estimator = total (trace) variance across its components,
        # averaged over parameters -> a single scalar "how noisy is the PG direction"
        grad_var[lam] = float(np.mean(np.var(G, axis=0)))
    return adv_var, grad_var


print("training a light, shared snapshot to freeze ...")
frozen_policy, frozen_value = freeze_lightly_trained(seed=0, light_iters=4)
print("measuring estimator variance across many independent rollouts per lambda ...")
adv_var, grad_var = advantage_variance_vs_lambda(frozen_policy, frozen_value)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
lams = LAMBDAS
ax[0].plot(lams, [adv_var[l] for l in lams], "o-", color="tab:purple", lw=2)
ax[0].set_xlabel(r"$\lambda$"); ax[0].set_ylabel("variance of mean advantage across rollouts")
ax[0].set_title("Advantage-estimate variance vs $\\lambda$"); ax[0].grid(alpha=0.3)
ax[1].plot(lams, [grad_var[l] for l in lams], "s-", color="tab:orange", lw=2)
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel("variance of policy-gradient vector across rollouts")
ax[1].set_title("Policy-gradient variance vs $\\lambda$"); ax[1].grid(alpha=0.3)
fig.suptitle("Direct measurement: estimator variance rises with $\\lambda$  (frozen policy, "
             f"{40} independent rollouts)", y=1.03)
plt.tight_layout(); plt.show()

print("advantage variance by lambda:", {l: round(adv_var[l], 5) for l in lams})
print("gradient  variance by lambda:", {l: round(grad_var[l], 6) for l in lams})
print("\n--> variance generally INCREASES with lambda: lambda=0 (one-step TD) is the steadiest,")
print("    lambda=1 (Monte-Carlo) is the noisiest. That is the bias/variance dial, measured.")


## 9. (Optional) Watch a trained agent

A greedy rollout from one of the trained sweet-spot policies, rendered as an inline GIF. Purely
for fun — skip on a headless machine. It should balance the pole far longer than a random agent.

In [ ]:
import io, base64
from IPython.display import HTML

def render_episode_frames(env_id, policy_fn, seed=0, max_steps=500):
    """Roll out one episode collecting rgb frames. `policy_fn(obs)->action`."""
    env = gym.make(env_id, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames = []
    for _ in range(max_steps):
        frames.append(env.render())
        a = policy_fn(obs)
        obs, r, term, trunc, _ = env.step(a)
        if term or trunc:
            break
    env.close()
    return frames

def frames_to_gif_html(frames, fps=30, scale=1.0):
    """Encode rgb frames -> an inline animated GIF (needs Pillow, bundled with matplotlib)."""
    try:
        from matplotlib import animation
        if not frames:
            return HTML("<i>(no frames)</i>")
        fig = plt.figure(figsize=(frames[0].shape[1] / 72 * scale,
                                   frames[0].shape[0] / 72 * scale))
        plt.axis("off")
        im = plt.imshow(frames[0])
        def _upd(i):
            im.set_data(frames[i]); return (im,)
        anim = animation.FuncAnimation(fig, _upd, frames=len(frames), interval=1000 / fps, blit=True)
        buf = io.BytesIO()
        anim.save(buf, writer="pillow", fps=fps)
        plt.close(fig)
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return HTML(f'<img src="data:image/gif;base64,{b64}"/>')
    except Exception as e:
        return HTML(f"<i>GIF rendering skipped: {e}</i>")

# train one fresh sweet-spot policy for the demo (lambda=0.95)
print("training a demo policy at lambda=0.95 ...")
_, demo_policy, _ = train(lmbda=0.95, seed=0)
def greedy(o):
    with torch.no_grad():
        dist = demo_policy(torch.as_tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0))
        return int(dist.probs.argmax(dim=1).item())   # deterministic = most-likely action

_frames = render_episode_frames(ENV_ID, greedy, seed=123)
print(f"trained agent survived {len(_frames)} steps")
frames_to_gif_html(_frames, fps=30)

## 10. Takeaways & things to try

**$\lambda$ is the bias$\leftrightarrow$variance dial of advantage estimation.**

- **$\lambda=0$** = one-step TD: $\hat A_t=\delta_t$. **Low variance, high bias** — it trusts the
  critic completely, so when $V$ is wrong (early training, or a hard-to-fit value surface) the
  advantages are systematically off and learning suffers.
- **$\lambda=1$** = Monte-Carlo: $\hat A_t=\sum_l\gamma^l r_{t+l}-V(s_t)$. **Unbiased** (modulo
  $\gamma$), but **high variance** — each estimate sums many noisy future rewards, so gradients
  are jittery and seed-to-seed spread is large.
- **$\lambda\approx0.9$–$0.97$**: the usual sweet spot. Enough bootstrapping to crush variance,
  little enough that the critic's bias stays small. Plot 1 typically shows fastest, most reliable
  learning here; Plot 2 shows variance climbing monotonically toward $\lambda=1$.

**Why it matters beyond CartPole.** GAE is the default advantage estimator inside modern PPO —
including the PPO used for RLHF / RL-on-LLMs. The same $\lambda$ knob is doing the same job there:
trading the bias of a learned value model against the variance of sampled trajectory returns.

**Things to try**
- Sweep a finer grid near the elbow, e.g. `LAMBDAS = [0.9, 0.92, 0.95, 0.97, 0.99]`.
- Cripple the critic (`PPO_VF_COEF = 0.0`, or `HIDDEN = 8`): a worse $V$ makes small-$\lambda$
  bias more damaging — the sweet spot drifts **up** toward Monte-Carlo.
- Lower $\gamma$ to 0.95: shorter effective horizon changes how much variance even $\lambda=1$
  accumulates.
- In Plot 2, raise `n_rollouts` for tighter variance estimates, or shrink `rollout_len` to see
  how horizon length amplifies the $\lambda=1$ variance.
- Turn `NORM_ADV = False` and watch the raw-advantage scale explode at $\lambda=1$.

**Theory companions (Notion):**
[PPO](https://app.notion.com/p/37f95008d76681ae994febc1f7e89aa5) &nbsp;·&nbsp;
[Part 3: Intro to Policy Optimization](https://app.notion.com/p/37895008d7668195b974f3c59beb74f6)